# AI Stock Market Assistant
**Assignment 1.1 — LangChain + LangGraph multi-agent system**

The implementation lives in two small modules next to this notebook, so the notebook and the deployed
Streamlit app (`app.py`) run exactly the same code:

* `core.py` — Groq LLM (LangChain `ChatGroq`), stock-data tool (yfinance), NewsAPI tool, minimal RAG
  (documents → text splitter → FastEmbed → `InMemoryVectorStore` → retrieval), prompt + chain.
* `graph.py` — LangGraph shared state, specialised agent nodes, conditional routing, and the recommendation agent.

This notebook walks through every component, shows the relevant source, and runs the full workflow.

## Assignment objective and where each requirement is met

| Requirement | Where |
|---|---|
| Analyse stock data | §4 stock tool, `stock_analysis` node (§9) |
| Financial news with RAG | §5–6 NewsAPI + RAG, `news_rag` node |
| LLM insights | §7 chain, `insight` node |
| Simulated BUY / SELL / HOLD + justification | `recommendation` node (§9, §12–13) |
| LangChain: LLM, prompt templates, tools / chains / runnables | §3, §4, §5, §7, §11 |
| LangGraph: multi-agent, state, nodes, edges, conditional edge | §8–11 |

## 1. Imports

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import inspect
from IPython.display import Markdown, display
from langgraph.graph import END, START, StateGraph

from core import (analysis_prompt, build_analysis_chain, build_news_store, get_financial_news,
                  get_llm, get_stock_data, retrieve_news_context)
from graph import (DEFAULT_QUESTION, Recommendation, State, fallback_node, insight_agent,
                   news_rag_agent, recommendation_agent, recommendation_prompt,
                   route_after_stock_analysis, stock_analysis_agent)

C:\IIT_Roorkee Course\ai-stock-market-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Environment / configuration
Secrets come from `.env` (loaded in `core.py` with `python-dotenv`). Only presence is shown, never values.

In [2]:
for key in ("GROQ_API_KEY", "GROQ_MODEL", "NEWS_API_KEY"):
    print(f"{key}: {'set' if os.getenv(key) else 'MISSING'}")
print("Model:", os.getenv("GROQ_MODEL"))

GROQ_API_KEY: set
GROQ_MODEL: set
NEWS_API_KEY: set
Model: openai/gpt-oss-20b


## 3. LLM setup (LangChain `ChatGroq`)

In [3]:
llm = get_llm()
print(type(llm).__name__, "->", llm.invoke("Reply with the single word: ready").content)

ChatGroq -> ready


## 4. Stock-data tool (LangChain `@tool`, yfinance)

In [4]:
print("Tool:", get_stock_data.name, "-", get_stock_data.description.splitlines()[0])
stock = get_stock_data.invoke({"ticker": "AAPL"})
stock

Tool: get_stock_data - Fetch 6 months of daily prices for a ticker and return summary metrics.


{'ticker': 'AAPL',
 'last_close': 336.58,
 'change_1d_pct': 0.89,
 'change_5d_pct': 1.07,
 'change_1m_pct': 6.44,
 'change_6m_pct': 33.01,
 'sma_20': 334.47,
 'sma_50': 322.23,
 'high_6m': 345.34,
 'low_6m': 245.26,
 'daily_volatility_pct': 1.71,
 'avg_volume': 49564467,
 'n_sessions': 128}

In [5]:
# Invalid tickers return an error dict instead of raising
get_stock_data.invoke({"ticker": "NOTATICKERXYZ"})

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOTATICKERXYZ"}}}


$NOTATICKERXYZ: No data found, symbol may be delisted


{'ticker': 'NOTATICKERXYZ',
 'error': 'No (or too little) price data - check the ticker.'}

## 5. Financial-news retrieval (LangChain `@tool`, NewsAPI)

In [6]:
articles = get_financial_news.invoke({"query": "AAPL stock"})
print(f"{len(articles)} articles")
for a in articles[:5]:
    print(f"- [{a['source']}, {a['published_at'][:10]}] {a['title']}")

13 articles
- [Yahoo Entertainment, 2026-10-06] Paramount-Warner Bros. finally merger closes, creating media giant Skydance
- [Yahoo Entertainment, 2026-10-06] Paramount-Warner Bros. finally merger closes, creating media giant Skydance
- [Biztoc.com, 2026-10-05] NYSE parent company's joint venture plans 24/7 tokenized stock trading, and it's a huge deal
- [Biztoc.com, 2026-09-25] Apple Just Gained 10% in a Month. What Will It Take for AAPL Stock to Finally Hit $350?
- [Biztoc.com, 2026-10-06] Here's How Many Shares of Apple You'd Need for $10,000 in Yearly Dividends


## 6. Minimal RAG pipeline
`articles → Documents → RecursiveCharacterTextSplitter → FastEmbed embeddings → InMemoryVectorStore → similarity retrieval`

In [7]:
store = build_news_store(articles)
print(f"{len(articles)} articles -> {len(store.store)} embedded chunks")

question = "What recent news could affect Apple's stock price?"
news_context = retrieve_news_context(store, question, k=4)
print(news_context)

13 articles -> 13 embedded chunks
[Biztoc.com, 2026-09-25] Apple Just Gained 10% in a Month. What Will It Take for AAPL Stock to Finally Hit $350?. The post Apple Just Gained 10% in a Month. What Will It Take for AAPL Stock to Finally Hit $350? appeared first on 24/7 Wall St..
A steady monthlong climb has put a bigger round number on the radar for Apple (NASDAQ:AAPL) shareholders. Apple stock is up 10% o…

[Yahoo Entertainment, 2026-09-26] Apple (AAPL) Plans a Health App Overhaul. Can Better Insights Boost Device Sales?. On September 9, 2026, the Wall Street Journal reported that Apple Inc. (NASDAQ:AAPL) is undertaking a major overhaul of its Health app and Apple Watch strate...

[Biztoc.com, 2026-10-06] Here's How Many Shares of Apple You'd Need for $10,000 in Yearly Dividends. Apple (NASDAQ: AAPL) is recognized as one of the world's most elite businesses. And as a result of its success, investors have profited immensely. The consumer discretionary stock has soared 1,070% in the past 

## 7. LangChain prompt template and chain (`prompt | LLM | parser`)

In [8]:
print("Prompt variables:", analysis_prompt.input_variables)
chain = build_analysis_chain()
print(chain.invoke({"stock_data": stock, "news_context": news_context, "question": question}))

Prompt variables: ['news_context', 'question', 'stock_data']


Apple’s share price is on a clear uptrend, closing at $336.58—just above its 20‑day SMA of $334.47 and well within the 6‑month high of $345.34. The stock has gained 6.44% in the last month and 33.01% over six months, while daily volatility sits at 1.71 %.  

**Recent news themes that could move the stock**  
- **Strong monthly rally**: Apple has posted a 10 % gain in the last month, fueling speculation that it may hit the $350 threshold.  
- **Health‑app overhaul**: A major revamp of the Health app and Apple Watch strategy could broaden services revenue and drive device sales.  
- **Dividend outlook**: A discussion of how many shares are needed for $10,000 in yearly dividends highlights investor focus on yield.  
- **iPhone 18 Pro line**: A research firm’s projection of 6 million foldable iPhone sales underscores the importance of production capacity and product demand.  

**Risks**  
- Any slowdown in iPhone demand or supply‑chain constraints could temper growth.  
- The success of th

## 8. LangGraph shared state
One `TypedDict` is passed through every node; each agent returns only the keys it updates.

In [9]:
print(inspect.getsource(State))

class State(TypedDict, total=False):
    ticker: str
    question: str
    stock_data: dict
    news_context: str
    insights: str
    recommendation: str   # BUY | SELL | HOLD
    justification: str
    error: str



## 9. Specialised agent nodes
| Node | Role |
|---|---|
| `stock_analysis` | Stock Analysis Agent — fetch price data, add a rule-based trend label |
| `news_rag` | News/RAG Agent — NewsAPI → vector store → retrieval |
| `insight` | Insight Agent — LLM chain combining stock data and retrieved news |
| `recommendation` | Recommendation Agent — structured output: exactly one of BUY / SELL / HOLD + justification |
| `fallback` | Error path when no valid stock data is available |

In [10]:
for fn in (stock_analysis_agent, news_rag_agent, insight_agent):
    print(inspect.getsource(fn))

def stock_analysis_agent(state: State) -> dict:
    """Fetch price data and add a rule-based trend label."""
    data = get_stock_data.invoke({"ticker": state["ticker"]})
    if "error" in data:
        return {"stock_data": data, "error": data["error"]}
    sma50 = data["sma_50"]
    if sma50 is None:
        data["trend"] = "unknown (insufficient history)"
    elif data["last_close"] > data["sma_20"] > sma50:
        data["trend"] = "uptrend (close > SMA20 > SMA50)"
    elif data["last_close"] < data["sma_20"] < sma50:
        data["trend"] = "downtrend (close < SMA20 < SMA50)"
    else:
        data["trend"] = "mixed (no clear SMA alignment)"
    return {"stock_data": data}

def news_rag_agent(state: State) -> dict:
    """NewsAPI -> vector store -> similarity retrieval."""
    articles = get_financial_news.invoke({"query": f"{state['ticker']} stock"})
    store = build_news_store(articles)
    return {"news_context": retrieve_news_context(store, state["question"])}

def insight_age

In [11]:
print(inspect.getsource(Recommendation))
print(recommendation_prompt.messages[0].prompt.template)
print()
print(inspect.getsource(recommendation_agent))
print(inspect.getsource(fallback_node))

class Recommendation(BaseModel):
    action: Literal["BUY", "SELL", "HOLD"] = Field(description="Simulated recommendation")
    justification: str = Field(description="2-3 sentences citing only the provided data")

You produce SIMULATED, educational stock recommendations (not financial advice). Choose exactly one of BUY, SELL or HOLD. Base the decision and justification only on the stock data, news context and insights provided; do not add outside facts. Choose HOLD when the evidence is mixed or thin.

def recommendation_agent(state: State) -> dict:
    chain = recommendation_prompt | get_llm().with_structured_output(Recommendation)
    rec = chain.invoke({"ticker": state["ticker"], "stock_data": state["stock_data"],
                        "news_context": state["news_context"], "insights": state["insights"]})
    return {"recommendation": rec.action, "justification": rec.justification}

def fallback_node(state: State) -> dict:
    return {"error": f"Cannot analyse '{state['ticker']}':

## 10. Conditional routing
After `stock_analysis`, the graph continues to the news/RAG agent only if valid price data was retrieved;
otherwise it routes to `fallback` and ends, so no news or LLM calls are wasted on an invalid ticker.

In [12]:
print(inspect.getsource(route_after_stock_analysis))

def route_after_stock_analysis(state: State) -> Literal["news_rag", "fallback"]:
    """Continue only if valid price data was retrieved."""
    return "fallback" if state.get("error") else "news_rag"



## 11. Graph construction and compilation

In [13]:
g = StateGraph(State)
g.add_node("stock_analysis", stock_analysis_agent)
g.add_node("news_rag", news_rag_agent)
g.add_node("insight", insight_agent)
g.add_node("recommendation", recommendation_agent)
g.add_node("fallback", fallback_node)

g.add_edge(START, "stock_analysis")
g.add_conditional_edges("stock_analysis", route_after_stock_analysis)   # -> news_rag | fallback
g.add_edge("news_rag", "insight")
g.add_edge("insight", "recommendation")
g.add_edge("recommendation", END)
g.add_edge("fallback", END)

app = g.compile()
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	stock_analysis(stock_analysis)
	news_rag(news_rag)
	insight(insight)
	recommendation(recommendation)
	fallback(fallback)
	__end__([<p>__end__</p>]):::last
	__start__ --> stock_analysis;
	insight --> recommendation;
	news_rag --> insight;
	stock_analysis -.-> fallback;
	stock_analysis -.-> news_rag;
	fallback --> __end__;
	recommendation --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 12. Example: valid stock (`AAPL`) — all agents run

In [14]:
result = app.invoke({"ticker": "AAPL", "question": DEFAULT_QUESTION})
print("Keys in final state:", list(result))
print("\nTREND:", result["stock_data"]["trend"])
print("\n--- Retrieved news context ---\n" + result["news_context"])
print("\n--- Insights ---\n" + result["insights"])

Keys in final state: ['ticker', 'question', 'stock_data', 'news_context', 'insights', 'recommendation', 'justification']

TREND: uptrend (close > SMA20 > SMA50)

--- Retrieved news context ---
[Biztoc.com, 2026-09-25] Apple Just Gained 10% in a Month. What Will It Take for AAPL Stock to Finally Hit $350?. The post Apple Just Gained 10% in a Month. What Will It Take for AAPL Stock to Finally Hit $350? appeared first on 24/7 Wall St..
A steady monthlong climb has put a bigger round number on the radar for Apple (NASDAQ:AAPL) shareholders. Apple stock is up 10% o…

[CNBC, 2026-09-28] Jim Cramer says these stocks can win even as oil and bond yields squeeze the market. Here's the stocks Jim Cramer said will be less impacted by interest rates and the Iran war. CNBC's Jim Cramer said that two persistent market foes are making stock investing increasingly difficult but that doesn't mean investors should head for the exits.
We're talking about oil prices and…

[Biztoc.com, 2026-10-05] NYSE pare

## 13. Final simulated recommendation

In [15]:
display(Markdown(f"### {result['recommendation']}\n\n{result['justification']}"))

### BUY

Apple’s price is firmly above both the 20‑day and 50‑day SMAs and has risen 33 % over the past six months, with a 6.44 % monthly gain and a 0.89 % daily uptick, confirming a clear uptrend. The recent 10 % month‑long rally and the announcement of 24/7 tokenized trading are positive catalysts that could further lift liquidity and momentum. While macro risks such as rising oil prices and bear‑market signals exist, the current technical strength and momentum outweigh these concerns for a BUY stance.

## 14. Example: invalid ticker — conditional edge routes to `fallback`

In [16]:
bad = app.invoke({"ticker": "NOTATICKERXYZ", "question": DEFAULT_QUESTION})
print("Keys in final state:", list(bad))
print("Error:", bad["error"])
print("News/insights/recommendation skipped:", not any(k in bad for k in ("news_context", "insights", "recommendation")))

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOTATICKERXYZ"}}}


$NOTATICKERXYZ: No data found, symbol may be delisted


Keys in final state: ['ticker', 'question', 'stock_data', 'error']
Error: Cannot analyse 'NOTATICKERXYZ': No (or too little) price data - check the ticker.
News/insights/recommendation skipped: True


## 15. Deployed application
**Public URL:** _to be added after deployment_ — `https://<your-app-url>`

The deployed Streamlit app (`app.py`) calls the same `graph.run_assistant` workflow shown above.